# Module 8: Inspectable Neo4j Memory

This notebook keeps one memory story deliberately small: persist a preference statement, recall it for the same actor in a fresh session, prove a second actor sees nothing, and inspect where the preference came from and which real `Hotel` it describes.

Two boundaries are explicit. Multi-tenant mode rejects memory writes that omit a user identifier, but the application must still authenticate actors and authorize session IDs. The library's semantic searches are store-wide in 0.5.0, so this demo does not use them as an isolation boundary. Its recall query starts at the selected `User`.

**Prerequisites:** Module 1 has created exactly one `Hotel` named `AnyCompany Cairo Nile View`; Neo4j credentials point at the correct database; and AWS credentials in `AWS_REGION` may invoke Titan Text Embeddings V2. This module uses the same Neo4j instance and credentials as Module 1, so the repo-root `.env` described in the top-level README covers it. `load_config` reads this folder's `.env` first, then the repo-root `.env`. Without credentials, every live cell skips cleanly.

In [ ]:
# At an AWS event: dependencies are pre-installed. Run this cell as-is.
# Self-paced: uncomment the line below first.
# !pip install "neo4j-agent-memory[bedrock]==0.5.0" boto3 python-dotenv

print("Environment ready")

## 1. Configure one isolated workshop run

Every actor and session identifier includes a short run ID. Rerunning the notebook therefore cannot append to an earlier transcript, while the shared `demo08-` prefix still lets `cleanup_memory.py` remove all Demo 08 runs. Each live cell opens and closes its own memory client so a later-cell failure cannot leak a connection.

In [ ]:
import uuid
from contextlib import asynccontextmanager

import boto3

from memory_helpers import (
    DEMO_ID_PREFIX,
    HERO_HOTEL_NAME,
    WORKSHOP_OWNER,
    build_memory_client,
    get_actor_preferences_for_hotel,
    link_preference_to_message_and_hotel,
    load_config,
    tag_demo_records,
)

RUN_ID = uuid.uuid4().hex[:8]
ACTOR_A = f"{DEMO_ID_PREFIX}{RUN_ID}-guest-alice"
ACTOR_B = f"{DEMO_ID_PREFIX}{RUN_ID}-guest-blake"
SESSION_A1 = f"{DEMO_ID_PREFIX}{RUN_ID}-session-a1"
SESSION_A2 = f"{DEMO_ID_PREFIX}{RUN_ID}-session-a2"
SESSION_B1 = f"{DEMO_ID_PREFIX}{RUN_ID}-session-b1"

try:
    config = load_config()
except RuntimeError as exc:
    config = None
    print(f"Neo4j is not configured: {exc}")

MEMORY_READY = (
    config is not None and boto3.Session().get_credentials() is not None
)

@asynccontextmanager
async def open_memory():
    memory = build_memory_client(config)
    await memory.connect()
    try:
        yield memory
    finally:
        await memory.close()

if MEMORY_READY:
    print(f"Run {RUN_ID}: Neo4j at {config.uri}, Bedrock in {config.region}.")
else:
    print("Not configured. Every live cell below will skip.")

## 2. Persist the preference statement

The core path uses fixture messages instead of another model call. That makes the memory behavior deterministic and keeps this module focused on Neo4j. The first query also fails with an actionable message if Module 1 did not create the expected hero Hotel.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        hotels = await memory.query.cypher(
            """
            CYPHER 25
            MATCH (h:Hotel {name: $hotel_name})
            RETURN h.name AS name
            """,
            {"hotel_name": HERO_HOTEL_NAME},
        )
        if len(hotels) != 1:
            raise RuntimeError(
                f"Module 1 must create exactly one Hotel named "
                f"{HERO_HOTEL_NAME!r}; found {len(hotels)}."
            )

        preference_source = await memory.short_term.add_message(
            SESSION_A1,
            "user",
            f"I loved staying at {HERO_HOTEL_NAME}. A room on a high "
            "floor away from the elevator is a must for me.",
            user_identifier=ACTOR_A,
            extraction_mode="skip",
        )
        await memory.short_term.add_message(
            SESSION_A1,
            "assistant",
            "I will remember that hotel and room preference.",
            user_identifier=ACTOR_A,
            extraction_mode="skip",
        )

    print(f"Stored two fixture messages in {SESSION_A1}.")

## 3. Write one explicit preference and its provenance

The library creates the actor-owned `Preference`. Two small workshop-owned relationships make the graph inspectable: `DERIVED_FROM` points to the exact source message and `ABOUT_HOTEL` points directly to the existing Hotel. No `Entity` label or memory property is added to any Hotel node.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        preference = await memory.long_term.add_preference(
            category=f"hotels-demo08-{RUN_ID}",
            preference=(
                f"Loves {HERO_HOTEL_NAME} and wants a room on a high "
                "floor away from the elevator."
            ),
            context=f"Workshop run {RUN_ID}, session {SESSION_A1}",
            user_identifier=ACTOR_A,
        )

    linked = link_preference_to_message_and_hotel(
        config,
        str(preference.id),
        str(preference_source.id),
        HERO_HOTEL_NAME,
    )
    assert linked, "preference, message, or unique hero Hotel was missing"
    print("Preference linked to its source message and the real Hotel.")

## 4. Recall in a fresh session and isolate a second actor

Actor A now starts `SESSION_A2`, making the cross-session proof visible. Recall uses an actor-anchored graph traversal, not the store-wide vector index. Actor B also gets a real session before the same traversal returns nothing. In production, the application must bind these actor and session IDs to authenticated callers.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        await memory.short_term.add_message(
            SESSION_A2,
            "user",
            "What hotel and room preference do you have for me?",
            user_identifier=ACTOR_A,
            extraction_mode="skip",
        )
        await memory.short_term.add_message(
            SESSION_B1,
            "user",
            "What hotel and room preference do you have for me?",
            user_identifier=ACTOR_B,
            extraction_mode="skip",
        )

    for_a = get_actor_preferences_for_hotel(
        config, ACTOR_A, HERO_HOTEL_NAME
    )
    for_b = get_actor_preferences_for_hotel(
        config, ACTOR_B, HERO_HOTEL_NAME
    )
    assert len(for_a) == 1, f"actor A expected one preference, got {len(for_a)}"
    assert not for_b, f"actor B unexpectedly saw {len(for_b)} preference(s)"
    print(f"Actor A in fresh session: {for_a[0]['preference']}")
    print("Actor B: no preference returned.")

## 5. Inspect the complete provenance path

One parameterized Cypher query now shows the actor, preference, source message and session, and canonical Hotel.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        rows = await memory.query.cypher(
            """
            CYPHER 25
            MATCH (u:User {identifier: $actor})
                  -[:HAS_PREFERENCE]->(p:Preference)
                  -[:DERIVED_FROM]->(m:Message)
                  <-[:HAS_MESSAGE]-(c:Conversation),
                  (p)-[:ABOUT_HOTEL]->(h:Hotel {name: $hotel_name})
            RETURN u.identifier AS actor,
                   p.preference AS preference,
                   m.content AS source_message,
                   c.session_id AS source_session,
                   h.name AS hotel
            """,
            {"actor": ACTOR_A, "hotel_name": HERO_HOTEL_NAME},
        )
    assert len(rows) == 1, f"expected one provenance path, got {len(rows)}"
    for key, value in rows[0].items():
        print(f"{key:15s} {value}")

## Mark this run for scoped cleanup

The IDs already carry the `demo08-` namespace. The ownership marker provides a second cleanup handle. Cleanup deletes only namespaced memory records, orphaned Demo 08 preferences, and workshop-owned relationships; it never changes a Hotel node.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    marked = tag_demo_records(
        config,
        session_ids=[SESSION_A1, SESSION_A2, SESSION_B1],
        user_identifiers=[ACTOR_A, ACTOR_B],
    )
    print(f"Marked {marked} record(s) with {WORKSHOP_OWNER!r}.")

## Choosing a memory architecture

| Dimension | AgentCore Memory (Module 7) | Neo4j graph memory (this module) |
|-----------|-----------------------------|----------------------------------|
| How memory is written | Managed extraction | Explicit application writes |
| When it is recallable | After asynchronous extraction | Immediately after the write |
| Inspectability | Retrieved through a service API | Queryable graph with source provenance |
| Domain linking | Separate from domain data | Workshop-owned edge to the real `Hotel` |
| Isolation | Actor namespaces managed by the service | Scoped writes and actor-anchored reads; application authorizes sessions |
| Operations | AWS operates the store | You operate Neo4j and the embedding contract |

Choose AgentCore Memory for managed extraction and operations. Choose graph memory when explicit writes, immediate visibility, provenance, and domain relationships matter.

To remove every Demo 08 run without touching Hotel nodes:

```bash
uv run --with-requirements requirements.txt python cleanup_memory.py
```